
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 문서 파싱과 청킹

## 서론

**검색 증강 생성(RAG)** 애플리케이션의 효과는 기본적으로 검색하는 데이터의 품질에 따라 제한됩니다. **임베딩을 생성** 하기 전에 PDF, HTML 파일, 이미지와 같은 구조화되지 않은 **원시 데이터(raw data)** 를 수집, 저장하고 **대형 언어 모델(LLM)** 이 해석할 수 있는 형식으로 변환해야 합니다. 본 강의에서는 **데이터브릭스 인텔리전스 플랫폼(Databricks Intelligence Platform)** 내의 데이터 준비 단계, 특히 저장을 위한 **델타 레이크(Delta Lake)** 와 거버넌스를 위한 **유니티 카탈로그(Unity Catalog)** 의 활용에 초점을 맞춥니다. 바이너리 파일에서 구조화된 텍스트를 추출하는 네이티브 **ai_parse_document 함수** 를 살펴보고, 기본적인 **고정 크기 분할** 을 넘어 문맥을 인식하는 **텍스트 청킹(chunking)** 의 핵심 전략을 알아봅니다.

## 수업 목표

* RAG의 비정형 데이터 저장에서 Delta Lake 및 Unity Catalog Volumes의 역할을 설명할 수 있습니다.  
* Databricks AI 함수, 특히 `ai_parse_document` 및 그림 설명(figure description)과 같은 v2.0의 기능을 설명할 수 있습니다. 
* Autoloader 및 Spark Declarative Pipelines(SDP)를 사용한 효율적인 데이터 수집 패턴을 식별할 수 있습니다.
* 고정 크기, 재귀적, 임베딩 기반의 의미론적(semantic) 청킹 전략을 비교하고 대조할 수 있습니다.
* Databricks에서 파싱 및 청킹에 사용되는 표준 도구와 프레임워크를 매핑할 수 있습니다.

## A. 데이터 저장 및 처리 아키텍처

아래는 데이터 수집 및 처리 개요 워크플로입니다. 일반적인 사용 사례에서—그리고 이 모듈에서—우리는 워크플로를 따라가며, 첫 세 단계에 집중합니다:

RAG 아키텍처에서 데이터 스토리지는 가공되지 않은 소스 파일과 가공을 거친 구조화된 텍스트를 모두 수용할 수 있어야 합니다. **델타 레이크(Delta Lake)** 는 모든 데이터 유형에 대해 ACID 트랜잭션과 버전 관리를 제공하며 통합 데이터 관리 계층의 역할을 합니다. 델타 테이블은 구조화된 데이터에 최적화되어 있지만, RAG 워크플로우는 일반적으로 PDF와 같은 비구조화된 파일에서 시작됩니다.

**유니티 카탈로그 볼륨(Unity Catalog Volumes)** 은 이러한 비정형 파일을 위한 거버넌스 계층을 제공합니다. 볼륨을 사용하면 테이블이나 모델에 적용되는 것과 동일한 통합 권한 모델을 통해 원본 파일에 대한 액세스를 관리할 수 있습니다. 원본 문서는 볼륨에 저장하고 가공된 텍스트는 델타 테이블에 저장함으로써, 최초 파일부터 검색에 사용되는 청크(chunk) 단위 텍스트에 이르기까지 전체 데이터 계보(lineage)를 완벽하게 유지할 수 있습니다.

**참고:** 볼륨에는 가공되지 않은 '원본' 파일(브론즈 레벨)이 저장되고, 델타 테이블에는 '파싱 및 청크 분할'이 완료된 텍스트(실버/골드 레벨)가 저장됩니다.

아래는 데이터 수집 및 처리 워크플로우의 개요입니다. 일반적인 활용 사례와 본 모듈에서는 다음과 같은 워크플로우를 따르며, 특히 처음 세 단계에 초점을 맞춥니다.

1. **데이터 수집 및 전처리:** Unity Catalog 볼륨에서 파일을 읽어온 후 AI 함수를 사용하여 파싱합니다.
1. **데이터 저장:** 파싱된 문서를 델타 레이크에 저장하고 필요한 거버넌스 제어를 적용합니다. (거버넌스에 대한 자세한 내용은 본 모듈에서 다루지 않습니다.)
1. **청크:** 임베딩 생성에 적합하도록 데이터를 작은 청크 단위로 분할합니다.

<!-- <img src="../Includes/images/02-process-diagram.png" alt="Data storage, ingestion and processing workflow" /> -->

![02-process-diagram](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-process-diagram.png)

*그림 1. 이 도표는 데이터 수집, 처리 및 임베딩 생성 워크플로의 5가지 주요 단계를 보여줍니다.* 


## B. AI 함수를 이용한 문서 처리

문서 처리는 검색 에이전트를 위한 고품질 지식 베이스를 구축하는 데 필수적이며, 특히 문서가 주요 지식 소스일 때 그 중요성이 더욱 커집니다. 실제 문서는 흔히 복잡한 구조를 가지고 있습니다. 이러한 문제를 해결하기 위해, 우리는 다양한 문서 형식에서 정보를 해석하고 추출하도록 특별히 설계된 대형 언어 모델(LLM) 및 OCR 기능이 탑재된 LLM과 같은 고급 모델을 활용합니다. 데이터브릭스(Databricks)는 이 프로세스를 간소화할 수 있도록 내장 AI 함수를 제공합니다. 특히 **ai_parse_document** 함수를 사용하면 PDF와 이미지를 강력하게 파싱하여 원본 파일에서 구조화된 콘텐츠와 레이아웃 정보를 직접 추출할 수 있습니다.

### B1. 문서 처리 과제

실제 문서를 분석(파싱)하는 일은 복잡합니다. 문서가 단순히 일반 텍스트로만 이루어진 경우는 거의 없기 때문입니다. 문서는 대개 **이미지, 다단 레이아웃, 표, 도표, 머리말, 본문 소제목, 페이지 번호** 등이 뒤섞인 형태를 띱니다. 이러한 정보의 의미와 맥락을 유지하면서 올바르게 추출하는 데는 다음과 같은 몇 가지 까다로운 과제가 따릅니다.

* **계층적 정보:** 차트나 다이어그램은 유기적인 계층 관계를 담고 있는 경우가 많아, 이를 그대로 보존하며 추출해야 합니다.  
* **순서 보존:** 다단 양식의 문서에서는 읽는 순서가 매우 중요합니다. 단순한 방식으로 분석하면 열과 열이 잘못 뒤섞여 합쳐질 수 있습니다.
* **맥락적 무결성:** 차트나 제품 사진 같은 이미지는 관련된 본문 설명과 서로 분리되지 않도록 긴밀하게 연결되어 있어야 합니다.



<!-- <img src="../Includes/images/02-complex-page-structure-example.png" alt="An example page showing complex page layout" /> -->
![02-complex-page-structure-example](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-complex-page-structure-example.png)

*그림 2. 이 이미지는 복잡한 페이지 레이아웃을 가진 예시 페이지 레이아웃을 보여줍니다* 




### B2. 구문 분석용 LLM과 OCR

이러한 문제를 해결하기 위해 현대적 접근법은 **대형 언어 모델(LLM)** 과 **광학 문자 인식(OCR)** 모델을 활용합니다. 전통적인 텍스트 파서와 달리, 이 모델들은 문서 레이아웃을 "볼 수 있습니다." OCR 모델은 이미지 내 텍스트를 식별할 수 있으며, 멀티모달 LLM은 그것 위 이미지에 캡션이 속하거나 테이블이 여러 페이지에 걸쳐 있음을 이해하여 요소의 공간적 배열을 해석할 수 있습니다.

### B3. `ai_parse_document` 사용

데이터브릭스(Databricks)는 **AI 함수**를 통해 이 과정을 간소화합니다. 개발자는 간단한 SQL 또는 파이썬(Python) 함수 호출만으로 이러한 고급 AI 모델을 데이터에 직접 적용할 수 있으며, 이 덕분에 모델 추론을 위한 별도의 인프라를 관리할 필요가 없습니다. 이 함수들은 서버리스 방식으로 실행되어 수백만 행을 처리할 수 있도록 자동으로 확장되며, 유니티 카탈로그(Unity Catalog) 내에서 거버넌스가 적용된 데이터에 직접 작동합니다.

이 작업에 가장 핵심적으로 사용되는 데이터브릭스 도구는 `ai_parse_document` 함수입니다. 이 함수는 최첨단 생성형 AI 모델을 호출하여 비정형 문서(PDF 및 이미지 등)에서 정형 콘텐츠를 추출하고, 그 결과를 정형 JSON 객체(VARIANT 타입)로 반환합니다.

**주요 기능 (스키마 v2.0):**

* **레이아웃 인식:** 문서의 콘텐츠와 레이아웃 정보를 분리합니다.  
* **그림 설명 생성:** PPDF 내에 포함된 차트와 이미지에 대한 텍스트 설명을 자동으로 생성하여, 대규모 언어 모델(LLM)이 시각적 데이터에 접근할 수 있도록 합니다.
* **바운딩 박스(Bounding Boxes):** 텍스트 요소의 좌표(bbox)를 반환하여, 사용자 인터페이스(UI)에서 출처를 강조 표시할 때 유용하게 활용할 수 있습니다.

**예제 구현:**

```sql
-- 이진 PDF 데이터에서 문서 레이아웃과 내용을 추출합니다  
SELECT ai_parse_document(content) as parsed_document  
FROM read_files(  
  '/Volumes/path/to/pdfs/',  
  format => 'binaryFile'  
)
```

## C. 데이터 정제 및 변환

문서를 파싱한 후에는 파싱된 내용을 정리하고 그것을 목표에 맞는 형식으로 변환해야 합니다.



### C1. 잡음 감소

텍스트를 청크(chunk) 단위로 나누기 전에, 검색 품질을 떨어뜨리는 불필요한 요소들을 제거하는 정제 작업이 필요합니다. 원본 데이터를 그대로 추출하면 헤더, 푸터, 페이지 번호 등이 그대로 포함되는 경우가 많은데, 이는 텍스트의 의미적 흐름을 방해할 수 있습니다. 따라서 정제 로직은 파싱 단계의 출력 결과물에 적용되어야 합니다. HTML 데이터의 경우, 과도한 서식 태그가 모델에 혼선을 줄 수 있지만, `ai_parse_document` 를 활용하면 HTML 형식의 표를 지능적으로 추출할 수 있습니다. 이를 통해 웹 페이지 파싱에 필수적인 표 구조를 그대로 유지함으로써, 표 데이터의 해석 가능성을 확보할 수 있습니다.

### C2. 메타데이터 추출 및 추가

효과적인 RAG(검색 증강 생성) 시스템은 벡터 유사도 검색을 수행하기 전에 메타데이터를 활용하여 검색 결과를 필터링합니다. 따라서 데이터 변환 과정에서 문서 제목, 저자 이름, 생성 날짜와 같은 메타데이터를 추출하고 연결하는 작업이 매우 중요합니다. 만약 이러한 메타데이터가 파일 속성에 포함되어 있지 않다면, **ai_extract** 와 같은 함수를 사용하여 비정형 텍스트로부터 structured field(예: '송장 발행일' 또는 '계약 유형')를 식별하고 추출할 수 있습니다.

## D. 청킹 전략
청킹(Chunking)은 긴 문서를 관리하기 쉬운 작은 단락으로 나누는 과정입니다. 임베딩 모델에는 컨텍스트 창(Context window)의 한계가 있고, 정확한 정보를 검색하려면 세밀한 검색 결과가 필요하기 때문에 이 단계는 필수적입니다.

또 다른 중요한 고려 사항은 컨텍스트 크기와 거대 언어 모델(LLM)의 성능 간의 관계입니다. '로스트 인 더 미들(Lost in the Middle)' 현상은 LLM이 긴 컨텍스트 창의 중간 부분에 묻힌 정보를 간과할 때 발생합니다. 따라서 중요한 세부 정보를 놓치지 않으려면 관련성 높은 작은 크기로 청크를 만드는 것이 좋습니다.

핵심 과제는 문서를 어떻게 가장 잘 나눌 것인가입니다. 청킹 방법에는 여러 가지가 있으며, 이 섹션에서는 가장 흔히 쓰이고 효과적인 접근 방식을 살펴보겠습니다.

**팁**: [ChunkViz](https://chunkviz.up.railway.app/)를 확인하여 청크 크기와 스플리터(Splitter)에 따라 청킹이 어떻게 이루어지는지 시각적으로 확인해 보세요.

### D1. 고정 크기 vs. 시맨틱 청킹(Fixed-Size vs. Semantic Chunking)

* **고정 크기 청크(기존/기준 방식):** 글자 수나 토큰 수(예: 500토큰)를 기준으로 텍스트를 엄격하게 분할합니다. 계산 비용은 적게 들지만, 문장이나 단락이 중간에 끊기는 경우가 많아 문맥이 훼손될 수 있습니다. 

* **시맨틱 청킹 (권장 표준 방식):** 임의로 글자 수를 맞추어 자르는 방식과 달리, **문장, 단락, 문서의 섹션** 등 의미 있는 언어적 경계를 기준으로 텍스트를 분할합니다. 문서의 논리적 구조를 존중함으로써 정보의 의미적 온전함을 보존합니다. 나아가 시맨틱 청킹은 각 청크에 관련 **메타데이터, 태그, 제목** 을 직접 주입하는 경우가 많기 때문에, 정보 검색 시 아주 짧은 텍스트 세그먼트라도 전체적인 맥락을 그대로 유지할 수 있도록 보장합니다.


<!-- <img src="../Includes/images/02-chunking-methods.png" alt="Chunking methods visualized"/> -->

![02-chunking-methods](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-chunking-methods.png)

*그림 3. 고정 크기 청킹과 의미론적 청킹의 시각적 표현입니다.*

### D2. 고급 청킹 전략

정보 검색 성능을 극대화하고 의미론적 일관성을 보장하려면, 복잡한 문서를 처리하고 경계 간의 맥락을 보존하기 위한 더욱 정교한 전략이 필요합니다.

* **청크 중복 (Chunk Overlap):** 이 기술은 연속된 청크 간의 중복 비율(예: 10~20%)을 정의합니다. 다음 청크의 시작 부분에 이전 텍스트의 일부를 반복함으로써, 청크 사이에서 맥락 정보가 손실되는 것을 방지하고 문장이나 아이디어가 갑자기 끊기는 현상을 막아줍니다.

* **윈도우 요약 (Windowed Summarization):** 각 청크에 이전 몇 개 청크의 '윈도우 요약'을 포함하는 '맥락 강화형' 청킹 방식입니다. 모델은 현재 텍스트만 보는 것이 아니라 이전 내용을 요약된 형태로 전달받으므로, 전체 이력을 모두 임베딩하는 비용을 들이지 않고도 더 넓은 맥락을 파악할 수 있습니다.

* **임베딩 기반 의미론적 청킹 (Embedding-Based Semantic Chunking):** 이 방법은 임베딩 모델을 사용하여 분할 지점을 결정하는 보다 발전된 방식입니다. 순차적인 문장 간의 의미론적 유사도를 계산하여, 주제가 크게 바뀔 때(즉, 유사도가 임계값 아래로 떨어질 때)에만 청크를 분할합니다. 이를 통해 각 청크가 명확하고 일관된 개념을 나타내도록 보장합니다.

<!-- <img src="../Includes/images/02-advanced-chunking-methods.png" alt="Advanced chunking methods visualized"/> -->

![02-advanced-chunking-methods](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/02-advanced-chunking-methods.png)

*그림 4. 고급 청킹 기법의 시각적 표현입니다. 각 색깔은 하나의 덩어리를 나타냅니다.* 


### D3. 모델 임베딩 고려사항

임베딩 모델은 이후 모듈에서 자세히 다루겠지만, 텍스트 분할(chunking) 단계인 지금 시점에서도 해당 모델의 기술적 제약을 반드시 고려해야 합니다.

* **컨텍스트 창 제한(Context Window Limits):** 모든 임베딩 모델에는 최대 토큰 제한(예: 512토큰, 8192토큰)이 있습니다. 텍스트 청크가 이 제한을 초과하면 모델은 제한을 벗어난 내용을 단순히 **잘라내어 무시**합니다. 이는 불완전한 벡터 표현과 데이터 손실로 이어집니다. 따라서 최대 청크 크기는 항상 임베딩 모델의 컨텍스트 창 제한보다 안전하게 작게 설정해야 합니다.

* **상세도와 컨텍스트의 절충(Granularity vs. Context):** 컨텍스트 창이 크면 더 큰 청크를 만들 수 있어 문맥을 더 많이 담아낼 수 있지만, 구체적인 세부 정보는 희석될 수 있습니다. 반면 컨텍스트 창이 작으면 청크도 작아질 수밖에 없어 표현은 더 정밀해지지만 주변 문맥이 누락될 수 있습니다. 청크 크기의 선택은 후속 과정에서 사용할 특정 임베딩 모델의 성능과 직접적으로 맞물려 있는 트레이드오프(상충 관계) 문제입니다.

## E. 청크용 도구 및 프레임워크

Databricks에서의 문서 처리는 기본 AI 함수와 LangChain 같은 주요 오픈소스 라이브러리를 결합하여 강력한 다단계 파이프라인을 구축합니다. 이 워크플로우는 순차적인 파싱과 청킹(분할) 과정을 통해 원시 파일을 임베딩 가능한 텍스트로 변환합니다.

1. **파싱(추출):** 첫 단계는 원시 파일을 파싱하고 레이아웃 정보와 함께 깨끗한 텍스트를 추출하는 것입니다.  
   * **`ai_parse_document` (네이티브):** 이 추천 도구는 표준 문서(PDF, 이미지)를 효율적으로 처리하며, OCR 및 레이아웃 분석을 서버리스 방식으로 수행합니다. 이후 후속 작업에 바로 사용할 수 있도록 구조화된 텍스트를 반환합니다.  
1. **청킹(분할):** 추출이 끝나면 텍스트를 관리하기 쉬운 작은 청크로 나누어야 합니다. 
   * **LangChain:** LangChain와 같은 라이브러리는 파싱된 텍스트에 대해 고급 분할 논리(예: `RecursiveCharacterTextSplitter` )를 제공합니다. LangChain의 텍스트 분할기 제품군은 다양한 형식과 전략을 지원하여 청킹 분야의 업계 표준으로 자리 잡고 있습니다. 
   * **사용자 정의 함수:** 개발자는 특정 마크다운 헤더를 기준으로 텍스트를 나누는 등 자신만의 특화된 분할 로직을 적용하기 위해, `ai_parse_document`의 결과물에 작동하는 사용자 정의 파이썬 사용자 정의 함수(UDF)를 구현할 수도 있습니다.

## F. 요약

Databricks에서 RAG(검색 증강 생성)를 위해 데이터를 준비하려면 데이터 수집, 파싱(분석), 변환으로 이어지는 안정적인 파이프라인이 필요합니다. 먼저, 원시 파일이 Unity Catalog 볼륨으로 수집됩니다. 그다음, 자체 제공되는 **`ai_parse_document` 함수** 를 사용해 이를 파싱합니다. 이 함수는 LLM과 OCR을 활용하여 PDF처럼 복잡한 문서에서도 깨끗한 텍스트와 레이아웃 정보를 추출해 냅니다. 마지막으로, 추출된 텍스트는 **재귀적 문자 분할(Recursive Character Splitting)** 이나 **임베딩 기반 의미론적 청킹(Embedding-Based Semantic Chunking)** 같은 고급 방식을 통해 전략적으로 청크(덩어리)로 나뉩니다. 이를 통해 임베딩 모델의 제한 사항을 준수하면서도, 검색 시스템이 정확하고 문맥이 풍부한 정보에 접근할 수 있도록 합니다.

**핵심 내용:**

1. **통합 거버넌스:** 원시 파일은 Unity Catalog 볼륨에, 처리된 청크는 Delta Table에 저장하여 전체 데이터의 이력(Lineage)과 보안을 철저히 유지합니다.
2. **순차 처리:** 문서 준비는 두 단계로 진행됩니다. 첫째, **`ai_parse_document`** 를 통해 안정적으로 데이터를 추출(OCR 및 레이아웃 분석)하고, 둘째, LangChain 같은 라이브러리를 활용해 논리적으로 문장을 분할합니다.
3. **고급 청킹 기술:** 단순히 고정된 크기로 자르는 방식에서 벗어나야 합니다. 문맥 손실을 방지하고 검색 정확도를 높이기 위해 **의미론적 분할 전략, 오버랩(중첩), 또는 부모 문서 검색(Parent Document Retrieval)** 방식을 도입하세요.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>